# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an end-to-end guide for loading and exploring the **FAIR²** dataset using the `mlcroissant` library. We'll walk through loading the Croissant dataset schema, overviewing record sets and fields by their `@id`, extracting records, and performing basic exploratory data analysis (EDA) and visualization.

### Dataset Source
The dataset schema (Croissant JSON-LD) URL:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and instantiate the Dataset object from the Croissant schema URL.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset with mlcroissant
dataset = mlc.Dataset(croissant_url)
print("Loaded dataset metadata:")
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
In this section, we enumerate available record sets, their `@id`s, and the fields/columns provided by each record set.

All references to dataset elements use their Croissant `@id`. We recommend copying these `@id`s for later use.

In [ ]:
# List available record sets (tables) and their fields by @id
if hasattr(dataset.metadata, "record_sets"):
    print("Record sets with @id and included fields:")
    for record_set in dataset.metadata.record_sets:
        print(f"- RecordSet @id: {record_set.id}")
        if hasattr(record_set, "fields"):
            for field in record_set.fields:
                print(f"    - Field @id: {field.id} | Column: {getattr(field, 'column', None)} | Name: {getattr(field, 'name', None)}")
        print()
else:
    print("[Warning] No record sets found in the dataset metadata. Attempting to infer via .records().")
    record_set_ids = dataset.record_sets
    print("Inferred record sets:", record_set_ids)

## 3. Data Extraction
Let's extract records from each available record set (by their `@id`). We'll load each into a pandas DataFrame, display the available columns and a preview of each table.

> ⚠️ To customize which record set(s) you want to explore, use their `@id` from above.


In [ ]:
# Collect all record set @ids
if hasattr(dataset, "record_sets") and len(dataset.record_sets) > 0:
    record_set_ids = dataset.record_sets
else:
    # Try inferring from metadata
    if hasattr(dataset.metadata, "record_sets"):
        record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
    else:
        # Fallback if not present
        record_set_ids = []

print("RecordSet @id list:", record_set_ids)

# Load all data into DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"--> Columns for {record_set_id}: {df.columns.tolist()}")
    display(df.head(2))

# Choose the main record set for EDA (the first by default if exists)
main_record_set_id = record_set_ids[0] if record_set_ids else None

## 4. Exploratory Data Analysis (EDA)
Perform basic EDA such as numeric filtering, normalization, and grouping for the main record set. All fields/columns are referenced by their `@id`---use the IDs you found above. If your dataset contains specific fields like 'molecular_weight' or 'abundance', substitute with the respective field `@id`.

In [ ]:
import numpy as np

# List numeric columns/fields by their exact @id or column name
numeric_candidate_columns = []
if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    # Heuristic: Pick float/int columns
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_candidate_columns.append(col)
    print("Available numeric columns for EDA (use their exact name or @id):", numeric_candidate_columns)

    if numeric_candidate_columns:
        numeric_field = numeric_candidate_columns[0]  # You can change this to your desired numeric @id
        threshold = df[numeric_field].mean() if np.issubdtype(df[numeric_field].dtype, np.number) else 0
        # Filter, normalize, and group data
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize (z-score)
        mean_val = filtered_df[numeric_field].mean()
        std_val = filtered_df[numeric_field].std()
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mean_val) / std_val
        print(f"Normalized {numeric_field} (z-score):")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Attempt to group by a categorical field
        group_field_candidates = [col for col in df.columns if df[col].dtype == object and df[col].nunique() < 10]
        if group_field_candidates:
            group_field = group_field_candidates[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped data by {group_field} (mean {numeric_field}):")
            display(grouped_df.head())
        else:
            group_field = None
            print('No suitable group_field detected (try categorical field with <10 unique values).')
    else:
        print('No numeric fields detected in the primary record set.')
else:
    print('No record set data loaded for EDA.')

## 5. Visualization
Now, let us visualize the distribution of the selected numeric field and, if available, how it varies across groups.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and main_record_set_id in dataframes and numeric_candidate_columns:
    numeric_field = numeric_candidate_columns[0]
    df = dataframes[main_record_set_id]
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df, showfliers=False)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()
else:
    print('Insufficient fields for visualization.')

## 6. Conclusion
In this notebook, we've loaded the FAIR² mass spectrometry dataset using the `mlcroissant` library, explored its structure using `@id` references for all entities, extracted data from available record sets, and conducted basic exploratory data analysis and visualization with pandas and seaborn.

For advanced analysis, consider consulting the dataset documentation, referencing specific `@id` fields for further feature engineering, and integrating with your downstream ML or statistical pipelines.